## Reranking 
- Reranking in RAG is the second-stage step where an additional model reorders the initially retrieved chunks so that the most truly relevant ones are placed at the top and sent to the LLM.

In [2]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("data.txt", encoding="utf8")
documents = loader.load()
print(f"Loaded {len(documents)} documents.")
print(len(documents[0].page_content))

Loaded 1 documents.
16645


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, 
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)
texts=text_splitter.split_documents(documents)

In [9]:
print(f"Split into {len(texts)} chunks.")
for i, chunk in enumerate(texts[:2]):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print(chunk.metadata)

Split into 106 chunks.
--- Chunk 1 ---
LangChain: Revolutionizing Language Model Applications
{'source': 'data.txt'}
--- Chunk 2 ---
LangChain is an open-source framework designed to simplify the development of applications powered by large language models (LLMs). Launched in 2022 by Harrison Chase, this innovative tool has
{'source': 'data.txt'}


In [16]:
text_embeddings = list(zip([t.page_content for t in texts], embeddings.embed_documents([t.page_content for t in texts])))
vectorstore = FAISS.from_embeddings(text_embeddings, embeddings, metadatas=[t.metadata for t in texts])
vectorstore.save_local("faiss_index")
print("Vector store saved locally to 'faiss_index' directory.")
print(f"Number of vectors in store: {vectorstore.index.ntotal}")  # Use ntotal for accurate count [page:1]


Vector store saved locally to 'faiss_index' directory.
Number of vectors in store: 106


In [28]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENROUTERKEY="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"
print(OPENROUTERKEY)
retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":3})


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


In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re

llm = ChatOpenAI(
    api_key=OPENROUTERKEY,
    base_url="https://openrouter.ai/api/v1",
    model="google/gemma-3-27b-it:free"
)

# Fix: Use ChatPromptTemplate.from_template()
prompt = ChatPromptTemplate.from_template("""
You will receive chunks of text from a document along with a user query. Your task is to rank the chunks based on their relevance to the query.

Query: {query}

Chunks:
{formatted_chunks}

Respond ONLY with comma-separated chunk numbers ranked from most to least relevant, e.g., "3,1,2,5,4". No other text.
""")

query = "What are the key components of LangChain?"
retrieved_docs = retriever.invoke(query)
formatted_chunks = "\n\n".join([f"Chunk {i+1}:\n{doc.page_content}" for i, doc in enumerate(retrieved_docs)])

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"query": query, "formatted_chunks": formatted_chunks})

# Parse ranks (e.g., "2,1,5" → [1,0,4])
rank_str = response.strip()
ranks = [int(x.strip()) - 1 for x in re.findall(r'\d+', rank_str)]  # Convert to 0-index
reranked_docs = [retrieved_docs[i] for i in ranks if 0 <= i < len(retrieved_docs)]

print("Reranked Chunks based on relevance to the query:")
for i, doc in enumerate(reranked_docs):
    print(f"{i+1}. {doc.page_content[:100]}...")


Reranked Chunks based on relevance to the query:
1. One of the key strengths of LangChain lies in its modular design. The framework is built around seve...
2. Let's delve deeper into the core components of LangChain. Chains, as mentioned earlier, are the buil...
3. One of the most compelling aspects of LangChain is its extensibility. The framework is designed to b...
